# La productivité a bel et bien accéléré · *Productivity really did accelerate*

Notebook compagnon du chapitre **13. IA, automatisation et productivité : un nouveau moteur de croissance ?** — [lire l'article](https://nmlab.io/ressources/ia-automatisation-productivite).
Companion notebook to chapter **13. AI, Automation and Productivity: A New Engine of Growth?** — [read the article](https://nmlab.io/en/ressources/ai-automation-and-productivity).

**Exécutez l'unique cellule ci-dessous** (bouton ▶ ou Ctrl+Entrée) : la figure se régénère avec les **données FRED du jour**. Passez `LANG = "en"` en tête de cellule pour les libellés anglais. — Run the single cell below (▶ or Ctrl+Enter) to rebuild the figure with **today's FRED data**; set `LANG = "en"` at the top for English labels.

Code : licence MIT · © 2026 [NMLab](https://nmlab.io) · dépôt [nmlab-finance/nmlab-figures](https://github.com/nmlab-finance/nmlab-figures)

In [ ]:
LANG = "fr"   # "fr" ou "en" — langue des libellés / label language

# Récupère puis active le style partagé NMLab (thème sombre + police Inter).
# Fetch and activate the shared NMLab style (dark theme + Inter font).
import urllib.request

urllib.request.urlretrieve("https://raw.githubusercontent.com/nmlab-finance/nmlab-figures/main/nmlab_style.py", "nmlab_style.py")
import nmlab_style as nm

nm.setup()


from pandas import Series

def load_productivity() -> Series:
    """Productivité horaire (OPHNFB, entreprises non agricoles) en glissement annuel.
    Labor productivity, year-over-year. pct_change(4) car la série est trimestrielle.
    U.S. labor productivity, live from FRED; year-over-year since the series is quarterly."""
    level = nm.load_fred("OPHNFB")
    return (level.pct_change(4) * 100).loc["2010":]

productivity = load_productivity()


import matplotlib.dates as mdates
import pandas as pd
from matplotlib.figure import Figure

PALE = "#c9d4e7"   # ligne « moyenne 2011-2019 », teinte pâle

LABELS = {
    "fr": dict(
        title="La productivité a bel et bien accéléré",
        sub="Production par heure travaillée, entreprises non agricoles américaines",
        ylab="productivité horaire, glissement annuel %",
        chatgpt="ChatGPT\nnov. 2022",
        note="Le rythme a plus que doublé depuis 2023. Mais l'accélération démarre quand moins de 10 % des grandes\n"
             "entreprises envisageaient encore d'utiliser l'IA : dater n'est pas attribuer. Source : BLS (OPHNFB)."),
    "en": dict(
        title="Productivity really did accelerate",
        sub="Output per hour worked, U.S. nonfarm business",
        ylab="labor productivity, year over year %",
        chatgpt="ChatGPT\nNov. 2022",
        note="The pace more than doubled since 2023. But the acceleration begins when fewer than 10% of large firms\n"
             "still planned to use AI: dating is not attributing. Source: BLS (OPHNFB)."),
}

def averages(productivity: Series) -> tuple[float, float]:
    """Moyennes de glissement annuel : 2011-2019, puis depuis 2023 / period averages."""
    return productivity.loc["2011":"2019"].mean(), productivity.loc["2023":].mean()

def avg_label(value: float, lang: str, period_fr: str, period_en: str) -> str:
    """Libellé « moyenne … : X,XX % » localisé (virgule décimale en français)."""
    if lang == "fr":
        return f"moyenne {period_fr} : {value:.2f} %".replace(".", ",")
    return f"{period_en} average: {value:.2f}%"

def build_figure(productivity: Series, lang: str) -> Figure:
    """Glissement annuel de la productivité, ses deux moyennes et le repère ChatGPT."""
    text = LABELS[lang]
    avg_pre, avg_post = averages(productivity)
    fig = nm.figure(height_px=1064)
    ax = nm.axes(fig)

    ax.plot(productivity.index, productivity, color=nm.COLORS["blue"], linewidth=2.9, zorder=3)
    ax.axhline(0, color=nm.COLORS["muted"], linewidth=1.7, alpha=0.9)

    # Repère vertical : sortie de ChatGPT (novembre 2022).
    chatgpt = pd.Timestamp("2022-11-01")
    ax.axvline(chatgpt, color=nm.COLORS["rose"], linestyle=(0, (1, 3)), linewidth=2.4)
    ax.text(pd.Timestamp("2022-06-01"), -2.55, text["chatgpt"], ha="right", va="center",
            fontsize=21, color=nm.COLORS["rose"], linespacing=1.4)

    # Moyenne 2011-2019 (pâle) et moyenne depuis 2023 (ambre).
    ax.plot([pd.Timestamp("2010-07-01"), pd.Timestamp("2020-03-01")], [avg_pre, avg_pre],
            color=PALE, linestyle=(0, (6, 4)), linewidth=3.2)
    ax.plot([pd.Timestamp("2023-01-01"), productivity.index[-1]], [avg_post, avg_post],
            color=nm.COLORS["amber"], linestyle=(0, (6, 4)), linewidth=3.4)
    ax.text(pd.Timestamp("2015-06-01"), 1.45,
            avg_label(avg_pre, lang, "2011-2019", "2011-2019"),
            ha="center", va="bottom", fontsize=22, fontweight="bold", color=PALE)
    if lang == "fr":
        post_lines = f"moyenne\ndepuis 2023 :\n{avg_post:.2f} %".replace(".", ",")
    else:
        post_lines = f"average\nsince 2023:\n{avg_post:.2f}%"
    ax.text(pd.Timestamp("2024-06-01"), 3.7, post_lines, ha="center", va="center",
            fontsize=23, fontweight="bold", color=nm.COLORS["amber"], linespacing=1.55)

    ax.set_ylim(-4, 6.2)
    ax.set_yticks(range(-4, 7, 2))
    ax.set_ylabel(text["ylab"])
    ax.set_xlim(pd.Timestamp("2010-01-01"), pd.Timestamp("2026-06-01"))
    ax.xaxis.set_major_locator(mdates.YearLocator(2, month=1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    nm.header(fig, text["title"], text["sub"])
    nm.footer(fig, text["note"])
    return fig

build_figure(productivity, LANG)